In [7]:
import os
import pandas as pd
from datetime import datetime

# Rutas
input_txt = r"M:\material_isa\estadisticos\_Turberas\day_in_off_snow.txt"
input_series_dir = r"M:\material_isa\estadisticos\_Turberas"
output_dir = r"M:\material_isa\estadisticos\_Turberas\indices_analisis_NDSI"
os.makedirs(output_dir, exist_ok=True)

# Leer y procesar el TXT
with open(input_txt, 'r', encoding='utf-8') as f:
    contenido = f.read()

# Separar bloques por turbera
bloques = contenido.split('✅ Procesado correctamente: ')[1:]

for bloque in bloques:
    lineas = bloque.strip().split('\n')
    nombre_raw = lineas[0].strip().replace('.csv', '')
    nombre = nombre_raw.replace('_periodo_nieve', '')

    # Cargar serie temporal de NDSI
    csv_path = os.path.join(input_series_dir, f"{nombre_raw}.csv")
    if not os.path.exists(csv_path):
        print(f"❌ No se encontró el archivo de serie NDSI: {csv_path}")
        continue

    serie = pd.read_csv(csv_path, sep=';', encoding='latin1')
    serie['FECHA'] = pd.to_datetime(serie['FECHA'], errors='coerce', dayfirst=True)
    serie = serie.dropna(subset=['FECHA', 'NDSI'])

    temporadas = []
    snow_ins = []
    snow_offs = []
    duraciones = []
    persistencias = []

    for linea in lineas:
        if "Snow in" in linea and "Snow off" in linea:
            partes = linea.split("Snow in: ")[1].split(" | Snow off: ")
            snow_in_str = partes[0].strip()
            snow_off_str = partes[1].strip()

            try:
                snow_in = datetime.strptime(snow_in_str, "%d/%m/%Y")
                snow_off = datetime.strptime(snow_off_str, "%d/%m/%Y")
            except ValueError:
                continue  # Saltar si hay error en fecha

            temporada = f"{snow_in.year}-{snow_off.year}"
            duracion = (snow_off - snow_in).days

            # Filtrar NDSI en el periodo
            subset = serie[(serie['FECHA'] >= snow_in) & (serie['FECHA'] <= snow_off)]
            persistencia = (subset['NDSI'] < 0.4).sum()

            temporadas.append(temporada)
            snow_ins.append(snow_in.strftime("%Y-%m-%d"))
            snow_offs.append(snow_off.strftime("%Y-%m-%d"))
            duraciones.append(duracion)
            persistencias.append(persistencia)

    # Crear DataFrame
    df = pd.DataFrame({
        "Temporada": temporadas,
        "Snow In": snow_ins,
        "Snow Off": snow_offs,
        "Periodo": duraciones,
        "Persistencia": persistencias
    })

    # Guardar CSV
    out_path = os.path.join(output_dir, f"{nombre}_analisis_NDSI.csv")
    df.to_csv(out_path, index=False, sep=';')
    print(f"✅ Guardado: {out_path}")

# Diccionario de rangos para interpretación
rangos_persistencia = [
    (0, 2, "Muy persistente", 6),
    (3, 4, "Persistente", 5),
    (5, 6, "Medianamente persistente", 4),
    (7, 8, "Poco persistente", 3),
    (9, 10, "Muy poco persistente", 2),
    (11, 12, "Casi nula persistencia", 1)
]

def interpretar_persistencia(valor):
    for minimo, maximo, categoria, valor_cat in rangos_persistencia:
        if minimo <= valor <= maximo:
            return categoria, valor_cat
    return "Sin categoría", 0  # En caso de que esté fuera de rango

# 🔁 Procesar cada CSV ya generado y agregar las nuevas columnas
for archivo in os.listdir(output_dir):
    if archivo.endswith("_analisis_NDSI.csv"):
        path_csv = os.path.join(output_dir, archivo)
        df = pd.read_csv(path_csv, sep=';')

        categorias = []
        valores_categoria = []

        for val in df["Persistencia"]:
            cat, val_cat = interpretar_persistencia(val)
            categorias.append(cat)
            valores_categoria.append(val_cat)

        df["Categoria Interpretativa"] = categorias
        df["Valor Categoria"] = valores_categoria

        df.to_csv(path_csv, index=False, sep=';')
        print(f"🔁 Actualizado: {path_csv}")



✅ Guardado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDSI\Bassa_Nera_analisis_NDSI.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDSI\Clots_de_Rialba_analisis_NDSI.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDSI\Estanyeres_analisis_NDSI.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDSI\Plan_deth_Horn_analisis_NDSI.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDSI\Pla_de_Sartari_analisis_NDSI.csv
✅ Guardado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDSI\Pleta_de_Molieres_analisis_NDSI.csv
🔁 Actualizado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDSI\Bassa_Nera_analisis_NDSI.csv
🔁 Actualizado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDSI\Clots_de_Rialba_analisis_NDSI.csv
🔁 Actualizado: M:\material_isa\estadisticos\_Turberas\indices_analisis_NDSI\Estanyeres_analisis_NDSI.csv
🔁 Actualizado: M:\material_isa\estadisticos\_Tur

In [7]:
import os
import pandas as pd
from datetime import datetime

# Rutas
input_txt = r"M:\material_isa\estadisticos\_Turberas\day_in_off_snow.txt"
input_series_dir = r"M:\material_isa\estadisticos\_Turberas"
output_dir = r"M:\material_isa\estadisticos\_Turberas\indices_analisis_NDSI"
os.makedirs(output_dir, exist_ok=True)

# Leer y procesar el TXT
with open(input_txt, 'r', encoding='utf-8') as f:
    contenido = f.read()

# Separar bloques por turbera
bloques = contenido.split('✅ Procesado correctamente: ')[1:]

for bloque in bloques:
    lineas = bloque.strip().split('\n')
    nombre_raw = lineas[0].strip().replace('.csv', '')
    nombre = nombre_raw.replace('_periodo_nieve', '')

    # Cargar serie temporal de NDSI
    csv_path = os.path.join(input_series_dir, f"{nombre_raw}.csv")
    if not os.path.exists(csv_path):
        print(f"❌ No se encontró el archivo de serie NDSI: {csv_path}")
        continue

    serie = pd.read_csv(csv_path, sep=';', encoding='latin1')
    serie['FECHA'] = pd.to_datetime(serie['FECHA'], errors='coerce', dayfirst=True)
    serie = serie.dropna(subset=['FECHA', 'NDSI'])

    temporadas = []
    snow_ins = []
    snow_offs = []
    duraciones = []
    persistencias = []
    porcentajes = []

    for linea in lineas:
        if "Snow in" in linea and "Snow off" in linea:
            partes = linea.split("Snow in: ")[1].split(" | Snow off: ")
            snow_in_str = partes[0].strip()
            snow_off_str = partes[1].strip()

            try:
                snow_in = datetime.strptime(snow_in_str, "%d/%m/%Y")
                snow_off = datetime.strptime(snow_off_str, "%d/%m/%Y")
            except ValueError:
                continue

            temporada = f"{snow_in.year}-{snow_off.year}"
            duracion = (snow_off - snow_in).days

            # Filtrar dentro del periodo
            subset = serie[(serie['FECHA'] >= snow_in) & (serie['FECHA'] <= snow_off)]

            total_imagenes = len(subset)
            dias_sin_nieve = (subset['NDSI'] < 0.4).sum()

            if total_imagenes > 0:
                porcentaje = round((1 - dias_sin_nieve / total_imagenes) * 100, 1)
            else:
                porcentaje = None  # No hay datos

            temporadas.append(temporada)
            snow_ins.append(snow_in.strftime("%Y-%m-%d"))
            snow_offs.append(snow_off.strftime("%Y-%m-%d"))
            duraciones.append(duracion)
            persistencias.append(dias_sin_nieve)
            porcentajes.append(porcentaje)

    # Crear DataFrame
    df = pd.DataFrame({
        "Temporada": temporadas,
        "Snow In": snow_ins,
        "Snow Off": snow_offs,
        "Periodo (dias)": duraciones,
        "Persistencia (conteo)": persistencias,
        "Persistencia (%)": porcentajes
    })

    # Guardar nuevo archivo
    out_path = os.path.join(output_dir, f"{nombre}_analisis_NDSI_v2.csv")
    df.to_csv(out_path, index=False, sep=';')
    print(f"✅ Generado: {os.path.basename(out_path)}")


✅ Generado: Bassa_Nera_analisis_NDSI_v2.csv
✅ Generado: Clots_de_Rialba_analisis_NDSI_v2.csv
✅ Generado: Estanyeres_analisis_NDSI_v2.csv
✅ Generado: Plan_deth_Horn_analisis_NDSI_v2.csv
✅ Generado: Pla_de_Sartari_analisis_NDSI_v2.csv
✅ Generado: Pleta_de_Molieres_analisis_NDSI_v2.csv
